In [2]:
dat = {
    "1PRW": {
        "resids": [
            (False, 5, 20),
            (True, 16, 35),
            (False, 10, 25),
            (True, 52, 71),
            (False, 5, 20),
        ],
        "chain": "A",
    },
    "1BCF": {
        "resids": [
            (False, 8, 15),
            (True, 92, 99),
            (False, 16, 30),
            (True, 123, 130),
            (False, 16, 30),
            (True, 47, 54),
            (False, 16, 30),
            (True, 18, 25),
            (False, 8, 15),
        ],
        "chain": "A",
    },
    "5TPN": {
        "resids": [(False, 10, 40), (True, 163, 181), (False, 10, 40)],
        "chain": "A",
    },
    "5IUS": {
        "resids": [
            (False, 0, 30),
            (True, 119, 140),
            (False, 15, 40),
            (True, 63, 82),
            (False, 0, 30),
        ],
        "chain": "A",
    },
    "3IXT": {
        "resids": [(False, 10, 40), (True, 254, 277), (False, 10, 40)],
        "chain": "P",
    },
    "5YUI": {
        "resids": [
            (False, 5, 30),
            (True, 93, 97),
            (False, 5, 20),
            (True, 118, 120),
            (False, 10, 35),
            (True, 198, 200),
            (False, 10, 30),
        ],
        "chain": "A",
    },
    "1QJG": {
        "resids": [
            (False, 10, 20),
            (True, 38, 38),
            (False, 15, 30),
            (True, 14, 14),
            (False, 15, 30),
            (True, 99, 99),
            (False, 10, 20),
        ],
        "chain": "A",
    },
    "1YCR": {
        "resids": [(False, 10, 40), (True, 19, 27), (False, 10, 40)],
        "chain": "B",
    },
    "2KL8": {
        "resids": [(True, 1, 7), (False, 20, 20), (True, 28, 79)],
        "chain": "A",
    },
    "7MRX_60": {
        "resids": [(False, 0, 38), (True, 25, 46), (False, 0, 38)],
        "chain": "B",
    },
    "7MRX_85": {
        "resids": [(False, 0, 63), (True, 25, 46), (False, 0, 63)],
        "chain": "B",
    },
    "7MRX_128": {
        "resids": [(False, 0, 122), (True, 25, 46), (False, 0, 122)],
        "chain": "B",
    },
    # Skipped 4JHW (multiple chains P & F in the row)
    "4ZYP": {
        "resids": [(False, 10, 40), (True, 242, 436), (False, 10, 40)],
        "chain": "A",
    },
    "5WN9": {
        "resids": [(False, 10, 40), (True, 170, 189), (False, 10, 40)],
        "chain": "A",
    },
    # Skipped 6WV1 (multiple chains E & A in the row)
    "5TRV_short": {
        "resids": [(False, 0, 35), (True, 45, 65), (False, 0, 35)],
        "chain": "A",
    },
    "5TRV_med": {
        "resids": [(False, 0, 65), (True, 45, 65), (False, 0, 65)],
        "chain": "A",
    },
    "5TRV_long": {
        "resids": [(False, 0, 95), (True, 45, 65), (False, 0, 95)],
        "chain": "A",
    },
    "6E6R_short": {
        "resids": [(False, 0, 35), (True, 23, 35), (False, 0, 35)],
        "chain": "A",
    },
    "6E6R_med": {
        "resids": [(False, 0, 95), (True, 23, 35), (False, 0, 95)],
        "chain": "A",
    },
    "6E6R_long": {
        "resids": [(False, 0, 95), (True, 23, 35), (False, 0, 95)],
        "chain": "A",
    },
    "6EXZ_short": {
        "resids": [(False, 0, 95), (True, 28, 42), (False, 0, 95)],
        "chain": "A",
    },
    "6EXZ_med": {
        "resids": [(False, 0, 95), (True, 28, 42), (False, 0, 95)],
        "chain": "A",
    },
    "6EXZ_long": {
        "resids": [(False, 0, 95), (True, 28, 42), (False, 0, 95)],
        "chain": "A",
    },
}

In [4]:
import requests

BASE_URL = "http://files.rcsb.org/download/"
STORE_DIR = "pdbs"

for pdb in dat.keys():

    id = pdb[:4]
    r = requests.get(BASE_URL + id + ".pdb")
    r.raise_for_status()
    with open(STORE_DIR + "/" + id + ".pdb", "w") as f:
        f.write(r.text)

In [5]:
import MDAnalysis as mda
from MDAnalysis.analysis.distances import distance_array
import numpy as np
import polars as pl

rows = []

for pdb, v in dat.items():

    u = mda.Universe(STORE_DIR + "/" + pdb[:4] + ".pdb")

    # res_calphas = u.select_atoms(f"segid {v['chain']} and name CA")

    # assert len(res_calphas) == len(u.select_atoms(f"segid {v['chain']}").residues), pdb

    resids = v["resids"]

    for i in range(len(resids) - 2):
        t1 = resids[i]
        t2 = resids[i + 1]
        t3 = resids[i + 2]

        if t1[0] and not t2[0] and t3[0]:

            end_t1 = u.select_atoms(f"segid {v['chain']} and resid {t1[2]} and name CA")
            start_t3 = u.select_atoms(
                f"segid {v['chain']} and resid {t3[1]} and name CA"
            )

            actual_dist = distance_array(end_t1.positions[0], start_t3.positions[0])[
                0, 0
            ]
            seq_range = np.array(list(t2[1:]))
            loop_range = seq_range * 3.6
            helical_range = seq_range * 1.5

            rows.append(
                {
                    "name": pdb,
                    "seq_range": list(seq_range),
                    "actual_dist": actual_dist,
                    "dist_range_loop": list(loop_range),
                    "dist_range_helix": list(helical_range),
                }
            )

df = pl.DataFrame(rows)

In [9]:
df

name,seq_range,actual_dist,dist_range_loop,dist_range_helix
str,list[i64],f64,list[f64],list[f64]
"""1PRW""","[10, 25]",13.896766,"[36.0, 90.0]","[15.0, 37.5]"
"""1BCF""","[16, 30]",12.545031,"[57.6, 108.0]","[24.0, 45.0]"
"""1BCF""","[16, 30]",7.419351,"[57.6, 108.0]","[24.0, 45.0]"
"""1BCF""","[16, 30]",7.089596,"[57.6, 108.0]","[24.0, 45.0]"
"""5IUS""","[15, 40]",22.561622,"[54.0, 144.0]","[22.5, 60.0]"
"""5YUI""","[5, 20]",8.904697,"[18.0, 72.0]","[7.5, 30.0]"
"""5YUI""","[10, 35]",11.931805,"[36.0, 126.0]","[15.0, 52.5]"
"""1QJG""","[15, 30]",14.817203,"[54.0, 108.0]","[22.5, 45.0]"
"""1QJG""","[15, 30]",11.259039,"[54.0, 108.0]","[22.5, 45.0]"


In [8]:
ratio_loop = df.with_columns(
    (pl.col("dist_range_loop") / pl.col("actual_dist")).alias("ratio_loop")
).with_columns(
    pl.col("ratio_loop").list.get(0).alias("lower_bound"),
    pl.col("ratio_loop").list.get(1).alias("upper_bound"),
)

ratio_loop.select("lower_bound", "upper_bound").describe()

statistic,lower_bound,upper_bound
str,f64,f64
"""count""",10.0,10.0
"""null_count""",0.0,0.0
"""mean""",4.230313,9.014522
"""std""",2.150272,3.673569
"""min""",2.021405,3.36052
"""25%""",2.590531,6.476327
"""50%""",3.644413,8.608986
"""75%""",4.796146,10.560012
"""max""",8.124582,15.233591


In [10]:
ratio_loop

name,seq_range,actual_dist,dist_range_loop,dist_range_helix,ratio_loop,lower_bound,upper_bound
str,list[i64],f64,list[f64],list[f64],list[f64],f64,f64
"""1PRW""","[10, 25]",13.896766,"[36.0, 90.0]","[15.0, 37.5]","[2.590531, 6.476327]",2.590531,6.476327
"""1BCF""","[16, 30]",12.545031,"[57.6, 108.0]","[24.0, 45.0]","[4.591459, 8.608986]",4.591459,8.608986
"""1BCF""","[16, 30]",7.419351,"[57.6, 108.0]","[24.0, 45.0]","[7.763482, 14.556529]",7.763482,14.556529
"""1BCF""","[16, 30]",7.089596,"[57.6, 108.0]","[24.0, 45.0]","[8.124582, 15.233591]",8.124582,15.233591
"""5IUS""","[15, 40]",22.561622,"[54.0, 144.0]","[22.5, 60.0]","[2.393445, 6.38252]",2.393445,6.38252
"""5YUI""","[5, 20]",8.904697,"[18.0, 72.0]","[7.5, 30.0]","[2.021405, 8.085621]",2.021405,8.085621
"""5YUI""","[10, 35]",11.931805,"[36.0, 126.0]","[15.0, 52.5]","[3.017146, 10.560012]",3.017146,10.560012
"""1QJG""","[15, 30]",14.817203,"[54.0, 108.0]","[22.5, 45.0]","[3.644413, 7.288825]",3.644413,7.288825
"""1QJG""","[15, 30]",11.259039,"[54.0, 108.0]","[22.5, 45.0]","[4.796146, 9.592293]",4.796146,9.592293
